In [0]:
def write_delta(
    df: DataFrame,
    base_path: str,
    table_name: str,
    merge_keys: list[str]
) -> None:
    """
    Escreve um DataFrame em formato Delta.

    Se o Delta não existir, cria a tabela.
    Caso exista, realiza MERGE (UPSERT).

    Parameters
    ----------
    df : DataFrame
        DataFrame que será gravado.

    base_path : str
        Caminho da camada (SILVER_PATH ou GOLD_PATH).

    table_name : str
        Nome da tabela.

    merge_keys : list[str]
        Colunas que identificam unicamente um registro.
    """

    path = f"{base_path}/{table_name}"

    # Delta ainda não existe
    if not DeltaTable.isDeltaTable(spark, path):

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .save(path)
        )

        return

    delta_table = DeltaTable.forPath(spark, path)

    merge_condition = " AND ".join(
        [
            f"target.{key} = source.{key}"
            for key in merge_keys
        ]
    )

    (
        delta_table.alias("target")
        .merge(
            df.alias("source"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )